# 4. Score video chunks with the VLM and prepare spatial features

For each AVI in `data/raw/videos/`, run the VideoMeta activity gate and object tracking. It writes `activity_clips/`, `activity_scores.json`, an `activity_scores.png` graph, annotated window MP4s, spatial feature files, and `prepared_windows.json` to `data/interim/<video-name>/`.

**How windows are chosen** (`videometa>=0.0.19`). The OpenCV motion gate is gone. The video is cut into fixed 10 s chunks that overlap their neighbour by 2 s (`0-10, 8-18, 16-26, ...`), so every chunk opens with the end of the previous one as context and an action on a boundary is whole in at least one chunk. Each chunk is written as a 2 FPS MP4 and shown to the same Qwen3-VL model that stage 5 uses for annotation, with a rubric asking for a 0-1 **activity score** from how many people and vehicles move, how much they move, and how many distinct actions or interactions occur. The model also returns the moving subjects, an event count and a one-sentence summary; all of it is stored per chunk in `activity_scores.json`, whether or not the chunk was kept.

**Threshold.** `SCORE_THRESHOLD = 0.3` keeps everything beyond a lone person or vehicle passing through. MEVA has no "walks" or "drives" activity, but it does have `person_texts_on_phone`, `person_reads_document`, `person_heavy_carry` and `person_sits_down`, which the rubric places in the 0.3-0.4 band, so a 0.5 cut would drop them together with the empty car parks. Set it to 0.5 to keep only chunks with a clear door, vehicle-entry, meeting or manoeuvre. Because every chunk's score is saved, a different threshold can be tried by editing the value and re-running with `OVERWRITE = True`; the model is re-run too, so use `RESCORE = False` to reuse a previous `activity_scores.json` and only redo the filtering and tracking.

**Crop settings.** The annotated window MP4 is cropped to the window's actors (people plus any vehicle that moves) before it is resized, so a person keeps near full size; the crop is stored as `crop_box` in `prepared_windows.json` and stage 5 passes it to the model.

`VIDEO_FILTER` matches a substring of the AVI file name, so a full video name selects exactly one video, a camera id selects a camera, and `None` selects everything. With `OVERWRITE = False` (the default), any video that already has a `prepared_windows.json` is skipped.

Import `videometa` from [TestPyPI](https://test.pypi.org/project/videometa/) only (`videometa>=0.0.19`). This notebook must run natively on Apple Silicon with `mlx-vlm` installed, like stage 5: the gate loads the vision model. MLX/Metal is not available in the Airflow Docker container.


In [4]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import mlx.core as mx
from videometa import (
    ActivityGateConfig,
    ActivityWindowFinder,
    DetectionConfig,
    LocalQwenActivityScorer,
    ObjectBoundaryExtractor,
    RelevantWindow,
    WindowSpatialFeatureJoiner,
    probe_video,
    resolve_video_source,
)

PROJECT_ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "notebooks").is_dir() and (parent / "annotation_pipeline").is_dir()
)
RAW_VIDEOS = PROJECT_ROOT / "data" / "raw" / "videos"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
MAX_VIDEOS = None  # Testing: only one clip. Set to None to process every match.
#VIDEO_FILTER = "2018-03-05.13-20-01.13-25-01.school.G328"  # Pilot: one video. A camera id such as "G328" selects every video of that camera; None selects all videos.
VIDEO_FILTER = None
OVERWRITE = False   # True: rebuild the filtered videos. False: skip videos that already have prepared_windows.json.
RESCORE = False     # False: reuse an existing activity_scores.json (skips the model) and only redo the threshold, tracking and joining.

# Vision model used to score the chunks. Keep it the same as MODEL_ID in notebook 05 so the gate
# and the annotator judge the footage the same way. ~5 GB of weights.
MODEL_ID = "mlx-community/Qwen3-VL-8B-Instruct-4bit"
# MODEL_ID = "mlx-community/Qwen3-VL-4B-Instruct-4bit"   # smaller and faster, for a comparison

# 0.3 keeps anything beyond a lone subject passing through (MEVA's phone / reading / carrying /
# sitting activities sit at 0.3-0.4 in the rubric). 0.5 keeps only clear door, vehicle-entry,
# meeting and manoeuvre chunks.
SCORE_THRESHOLD = 0.3

gate_config = ActivityGateConfig(
    chunk_seconds=10.0,
    overlap_seconds=2.0,
    score_threshold=SCORE_THRESHOLD,
    clip_size=(960, 540),      # MEVA people are small at 1080p; 960x540 keeps a distant figure readable
    clip_fps=2.0,              # 20 frames per 10 s chunk; must match LocalQwenActivityScorer(video_fps=...)
    max_total_seconds=None,    # set e.g. 200.0 to cap VLM annotation time per 300 s video, highest score first
)

mx.set_cache_limit(2 * 1024**3)
scorer = None  # Loaded lazily, only when a video actually needs scoring.


def get_scorer():
    global scorer
    if scorer is None:
        print(f"Loading local model: {MODEL_ID}", flush=True)
        scorer = LocalQwenActivityScorer(model_id=MODEL_ID, video_fps=gate_config.clip_fps, max_tokens=300)
    return scorer


def window_to_dict(window):
    return {
        "start_seconds": window.start_seconds,
        "end_seconds": window.end_seconds,
        "start_frame": window.start_frame,
        "end_frame": window.end_frame,
        "score": window.score,
        "is_relevant": window.is_relevant,
        "subjects": list(window.subjects),
        "event_count": window.event_count,
        "summary": window.summary,
        "clip_path": window.clip_path,
        "error": window.error,
    }


def window_from_dict(item):
    return RelevantWindow(
        start_seconds=item["start_seconds"],
        end_seconds=item["end_seconds"],
        start_frame=item["start_frame"],
        end_frame=item["end_frame"],
        score=item["score"],
        is_relevant=False,
        subjects=tuple(item.get("subjects", [])),
        event_count=item.get("event_count", 0),
        summary=item.get("summary", ""),
        clip_path=item.get("clip_path"),
        error=item.get("error"),
    )


# MEVA-compatible COCO class IDs. Set to None or [] to track every YOLO class.
meva_complete_classes = [
    # Actors & Vehicles
    0,   # person
    1,   # bicycle
    2,   # car
    3,   # motorcycle
    5,   # bus
    7,   # truck
    # Infrastructure & Animals
    9,   # traffic light
    10,  # fire hydrant
    11,  # stop sign
    12,  # parking meter
    13,  # bench
    16,  # dog
    # Carried Props & Bags
    24,  # backpack
    25,  # umbrella
    26,  # handbag
    28,  # suitcase
    39,  # bottle
    41,  # cup
    73,  # book
    # Tools & Sports Items
    32,  # sports ball
    34,  # baseball bat
    36,  # skateboard
    43,  # knife
    # Furniture & Screens
    56,  # chair
    57,  # couch
    60,  # dining table (desks / work surfaces)
    62,  # tv (monitors / screens)
    # Electronics
    63,  # laptop
    64,  # mouse
    66,  # keyboard
    67,  # cell phone
]
boundary_extractor = ObjectBoundaryExtractor(
    DetectionConfig(
        model_path="yolo26l.pt",
        confidence_threshold=0.25,
        spatial_grid=3,
        classes=meva_complete_classes,
    )
)

video_paths = sorted(RAW_VIDEOS.glob("*.avi"))
if VIDEO_FILTER:
    video_paths = [path for path in video_paths if VIDEO_FILTER in path.name]
video_paths = video_paths[:MAX_VIDEOS]
if not video_paths:
    raise FileNotFoundError(f"No AVI files found in {RAW_VIDEOS} matching {VIDEO_FILTER!r}")

for video_path in video_paths:
    video_name = video_path.stem.removesuffix(".r13")
    output_dir = INTERIM_DIR / video_name
    manifest_path = output_dir / "prepared_windows.json"
    scores_path = output_dir / "activity_scores.json"
    if not OVERWRITE and manifest_path.exists():
        print(f"Skipping {video_name}: prepared_windows.json already exists (OVERWRITE=False)")
        continue
    frames_dir = output_dir / "window_frames"
    output_dir.mkdir(parents=True, exist_ok=True)

    source_path = resolve_video_source(video_path)

    # --- 1. Score every 10 s chunk with the VLM (or reuse the scores from a previous run).
    if not RESCORE and scores_path.exists():
        print(f"{video_name}: reusing {scores_path.name} (RESCORE=False)")
        video_info = probe_video(source_path)
        scored = [window_from_dict(item) for item in json.loads(scores_path.read_text(encoding="utf-8"))["chunks"]]
        finder = ActivityWindowFinder(scorer=None, config=gate_config)
    else:
        finder = ActivityWindowFinder(get_scorer(), gate_config, clip_directory=output_dir / "activity_clips")
        video_info, scored = finder.score_video(source_path)

    # --- 2. Filter: keep the chunks at or above the threshold (and under the budget, if any).
    windows = finder.build_windows(scored)
    selected_windows = [window for window in windows if window.is_relevant]
    failed = [window for window in windows if window.error]
    scores_path.write_text(
        json.dumps(
            {
                "video": video_name,
                "model_id": MODEL_ID,
                "chunk_seconds": gate_config.chunk_seconds,
                "overlap_seconds": gate_config.overlap_seconds,
                "score_threshold": gate_config.score_threshold,
                "chunks": [window_to_dict(window) for window in windows],
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    print(
        f"{video_name}: {len(selected_windows)} of {len(windows)} chunks kept at score >= {gate_config.score_threshold}"
        + (f", {len(failed)} failed to score" if failed else "")
    )

    figure, axis = plt.subplots(figsize=(14, 4))
    centres = [(window.start_seconds + window.end_seconds) / 2 for window in windows]
    axis.bar(
        centres,
        [window.score for window in windows],
        width=[window.duration_seconds * 0.8 for window in windows],
        color=["seagreen" if window.is_relevant else ("crimson" if window.error else "lightgray") for window in windows],
        edgecolor="white",
    )
    axis.axhline(gate_config.score_threshold, color="darkorange", linestyle="--", label=f"threshold {gate_config.score_threshold}")
    axis.set(
        title=f"VLM activity score per {gate_config.chunk_seconds:.0f}s chunk: {video_name}",
        xlabel="Time (seconds)",
        ylabel="Activity score",
        ylim=(0, 1),
    )
    axis.legend(loc="upper right")
    figure.savefig(output_dir / "activity_scores.png", dpi=150, bbox_inches="tight")
    plt.close(figure)

    # --- 3. Track objects in the kept chunks and join the spatial features, unchanged from before.
    object_annotations = boundary_extractor.extract(source_path, selected_windows, video_info.fps)

    joiner = WindowSpatialFeatureJoiner(
        output_directory=frames_dir,
        annotated_video_size=(1280, 720),
        annotated_video_fps=2,          # must match video_fps in notebook 05
        annotated_video_max_frames=24,
        encode_frame_images=False,
        crop_to_activity=True,          # crop to the actors (people + moving vehicles) before resizing
    )
    prepared_windows = [joiner.prepare(source_path, annotation) for annotation in object_annotations]
    scores_by_key = {(window.start_seconds, window.end_seconds): window for window in selected_windows}
    manifest = [
        {
            "start_seconds": item.start_seconds,
            "end_seconds": item.end_seconds,
            "activity_score": scores_by_key[(item.start_seconds, item.end_seconds)].score,
            "activity_summary": scores_by_key[(item.start_seconds, item.end_seconds)].summary,
            "object_features": item.object_features,
            "frame_features": item.frame_features,
            "image_messages": item.image_messages,
            "artifact_directory": item.artifact_directory,
            "annotated_video_path": item.annotated_video_path,
            "crop_box": item.crop_box,
        }
        for item in prepared_windows
    ]
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print(f"Prepared {len(prepared_windows)} windows for {video_name}")


Skipping 2018-03-05.13-15-00.13-20-00.bus.G340: prepared_windows.json already exists (OVERWRITE=False)
Skipping 2018-03-05.13-15-00.13-20-00.bus.G505: prepared_windows.json already exists (OVERWRITE=False)
Skipping 2018-03-05.13-15-00.13-20-00.bus.G506: prepared_windows.json already exists (OVERWRITE=False)
Skipping 2018-03-05.13-15-00.13-20-00.hospital.G341: prepared_windows.json already exists (OVERWRITE=False)
Skipping 2018-03-05.13-15-00.13-20-00.school.G423: prepared_windows.json already exists (OVERWRITE=False)
Skipping 2018-03-05.13-15-00.13-20-00.school.G424: prepared_windows.json already exists (OVERWRITE=False)
Skipping 2018-03-05.13-15-01.13-20-01.school.G328: prepared_windows.json already exists (OVERWRITE=False)
Skipping 2018-03-05.13-20-00.13-25-00.bus.G340: prepared_windows.json already exists (OVERWRITE=False)
Skipping 2018-03-05.13-20-00.13-25-00.school.G336: prepared_windows.json already exists (OVERWRITE=False)
Skipping 2018-03-05.13-20-00.13-25-00.school.G423: prepa